# Notebook 02 — Baseline BM25

**Disciplina:** Inteligência Artificial — FACOM/UFMS — 2026/1
**Tema:** IA para Reconhecimento de Vocalizações de Psittacidae

Implementa o **retriever BM25Okapi** como baseline obrigatório.
BM25 (Best Match 25) é derivado do modelo probabilístico de Robertson & Sparck Jones (1994),
generalizando o TF-IDF com saturação de frequência (parâmetro *k₁*) e normalização de comprimento (*b*).


In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, "..")

from src.utils import load_corpus, load_queries, write_trec_run
from src.preprocessing import tokenize, doc_text
from src.retrievers import BM25Retriever


## 1. Carregamento do corpus

In [2]:
docs = load_corpus("../data/corpus.jsonl")
print(f"Documentos: {len(docs)}")
print("Exemplo:", docs[0]["title"][:80])


Documentos: 1092
Exemplo: Decoding Insect Song: A Multitask Semisupervised Orthoptera Bioacoustic Classifi


## 2. Índice BM25

Hiperparâmetros padrão da literatura:
- **k₁ = 1.5** — controla a saturação de frequência do termo
- **b = 0.75** — controla a normalização pelo comprimento do documento

Ambos foram validados empiricamente em coleções de grande escala (Robertson et al., 1999).
O módulo M4 poderia otimizá-los via grid search sobre as qrels.


In [3]:
retriever = BM25Retriever(docs, k1=1.5, b=0.75)
print("Vocabulário BM25:", len(retriever.index.idf), "termos")


Vocabulário BM25: 12094 termos


## 3. Busca de exemplo

In [4]:
query = "deep learning for parrot call recognition mel spectrogram"
results = retriever.search(query, k=10)

print(f"Top-10 para: {query!r}\n")
for rank, (doc_id, score) in enumerate(results, 1):
    doc = next(d for d in docs if d["arxiv_id"] == doc_id)
    print(f"{rank:2}. [{score:6.3f}] {doc['title'][:75]}")
    print(f"    {doc_id} | {doc.get('primary_category','')} | {doc.get('published','')}")


Top-10 para: 'deep learning for parrot call recognition mel spectrogram'

 1. [12.533] Classification of Sound using Convolutional Neural Networks
    W4360585215 | Music and Audio Processing | 2022
 2. [11.701] Vocal individuality, but not stability, in wild palm cockatoos (<i>Probosci
    W2577418634 | Animal Vocal Communication and Behavior | 2017
 3. [11.400] A Novel Deep Learning Approach for Classification of Bird Sound Using Mel F
    W4402679966 | Animal Vocal Communication and Behavior | 2024
 4. [11.053] Accounting for both automated recording unit detection space and signal rec
    W4200109913 | Animal Vocal Communication and Behavior | 2021
 5. [10.818] Contact calls of island Brown-throated Parakeets exhibit both character and
    W4200606917 | Animal Vocal Communication and Behavior | 2021
 6. [10.583] Detection of ground parrot vocalisation: A multiple instance learning appro
    W2750994052 | Music and Audio Processing | 2017
 7. [10.457] Automatic Classification of Bir

## 4. Geração do arquivo de run (formato TREC)

In [5]:
queries = load_queries("../eval/queries.tsv")
runs_dir = Path("runs")
runs_dir.mkdir(exist_ok=True)

results_all = {}
for _, row in queries.iterrows():
    results_all[row["qid"]] = retriever.search(row["text"], k=100)

write_trec_run(results_all, runs_dir / "bm25.trec", system_name="bm25")
print("Run salva em runs/bm25.trec")

# Preview das 5 primeiras linhas
with open(runs_dir / "bm25.trec") as f:
    for i, line in enumerate(f):
        if i >= 5: break
        print(line, end="")


Run salva em runs/bm25.trec
q01 Q0 W4200109913 1 11.053212 bm25
q01 Q0 W2750994052 2 10.583058 bm25
q01 Q0 W2577418634 3 9.754332 bm25
q01 Q0 W2996427677 4 9.527878 bm25
q01 Q0 W4200606917 5 9.228867 bm25


## 5. Análise qualitativa — duas queries

Análise de acertos e falhas para motivar o desenvolvimento de retrievers mais sofisticados.


In [6]:
def show_top5(qid, query_text):
    print(f"\n=== {qid}: {query_text!r} ===")
    for rank, (doc_id, score) in enumerate(retriever.search(query_text, k=5), 1):
        doc = next(d for d in docs if d["arxiv_id"] == doc_id)
        print(f"  {rank}. [{score:.3f}] {doc['title'][:80]}")
        print(f"       {doc['abstract'][:200]}...")

# Query q01: específica ao tema → espera resultados de vocalizações de papagaios
show_top5("q01", "deep learning for parrot call recognition")
# Query q11: bioacústica geral → avalia cobertura do corpus
show_top5("q11", "bioacoustic event detection deep learning")



=== q01: 'deep learning for parrot call recognition' ===
  1. [11.053] Accounting for both automated recording unit detection space and signal recognit
       Abstract Research into the suitability of autonomous recording units (ARUs) when surveying for vocal species is increasing. Simultaneously, there has been extensive research into methods for efficient...
  2. [10.583] Detection of ground parrot vocalisation: A multiple instance learning approach
       Ground parrot vocalisation can be considered as an audio event. Test-based diverse density multiple instance learning (TB-DD-MIL) is proposed for detecting this event in audio files recorded in the fi...
  3. [9.754] Vocal individuality, but not stability, in wild palm cockatoos (<i>Probosciger a
       The ability to identify individuals within a population is often essential for a detailed understanding of the ecology and conservation of a species. However, some species, including large parrots, ar...
  4. [9.528] Individual sig

## 6. Discussão

**Conexão com o paradigma probabilístico:**
O BM25 estima P(relevante | query, documento) via TF-IDF ponderado e normalização de comprimento,
derivando-se do modelo de linguagem de Robertson & Sparck Jones.
Em relação ao KNN denso (notebook 03), o BM25 tende a ter alta precisão para termos técnicos raros
(e.g., "psittacidae", "mel spectrogram") mas falha em sinonímia semântica
(e.g., "parrot call" ≠ "psittacine vocalization").

**Limitações observadas:**
- Não captura semântica (busca lexical)
- Sensível a variações ortográficas e siglas
- Hiperparâmetros podem ser otimizados (módulo M4)
